# Calibration across esports games
### Polymarket · Advanced Applied Statistics

**Research question.** How do calibration and probabilistic forecast errors vary across the videogames represented in closed Polymarket esports markets?

We use the existing esports-labelled subset and inspect the **event slug and title** to assign a game. These labels are deliberately conservative; events that cannot be assigned confidently remain unclassified. The starting esports category is itself inferred, so true esports events classified elsewhere may be missing.

**Design.** Use all available hourly price observations from hours **2–24 before the market `endDate`**. A market contributes at most once to each 10-percentage-point probability bin, even if multiple hours fall in that bin; the same market can contribute to other bins. This trajectory-selected, market–bin-weighted dataset is retrospective, not a fixed-horizon trading backtest.

## 1 · Reproducible setup

Run from the repository root or set `POLYCATS_ROOT` before starting Jupyter. The notebook reads the 23-hour cache created by `Polymarket_Advanced_Statistics.ipynb`, plus the saved event records. It does not call an API. Set `POLYCATS_BOOTSTRAPS` to increase the default 500 event-bootstrap replicates.

In [ ]:
import os, json, hashlib, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.stats import false_discovery_control
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display

ROOT = Path(os.environ.get('POLYCATS_ROOT', Path.cwd())).expanduser().resolve()
DATA = ROOT / 'Data' / 'all_closed_sports'
MASTER = DATA / 'closed_sports_all_data.csv'
LINKS = DATA / 'market_events.jsonl'
EVENTS = DATA / 'events.jsonl'
PIPELINE = ROOT / 'Polymarket_Closed_Sports_Combined_Pipeline.ipynb'
PLOT_CODE = ROOT / 'plot_event_sport_calibration.py'
N_BINS, FIRST_HOUR, LAST_HOUR = 10, 2, 24
N_BOOT = int(os.environ.get('POLYCATS_BOOTSTRAPS', '500'))
MIN_EVENTS, MIN_MARKETS = 30, 100
SEED, EPS = 2026, 1e-6

for path in [MASTER, LINKS, EVENTS, PIPELINE, PLOT_CODE]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}. Open from the repository root or set POLYCATS_ROOT.')
if N_BOOT < 100:
    raise ValueError('Use at least 100 bootstrap replicates; 500 or more is preferable.')

def source_fingerprint(paths):
    return {str(p): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns} for p in paths}

source_files = [MASTER, LINKS, EVENTS, PIPELINE, PLOT_CODE]
fingerprint = source_fingerprint(source_files)
cache_key = hashlib.sha256(json.dumps({'files': fingerprint, 'bins': N_BINS, 'first_hour': FIRST_HOUR}, sort_keys=True).encode()).hexdigest()[:16]
CACHE = DATA / 'analysis_cache' / f'course_hourly_{cache_key}.pkl'
if not CACHE.is_file():
    raise FileNotFoundError(f'No current 23-hour analysis cache: {CACHE}. Run Polymarket_Advanced_Statistics.ipynb once to prepare it.')

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.family': 'serif',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .18, 'legend.frameon': False,
                     'axes.titlesize': 11, 'axes.labelsize': 10})
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print(f'Project: {ROOT}\nCache: {CACHE.name}\nHours: {FIRST_HOUR}–{LAST_HOUR} | bins: {N_BINS} | bootstrap: {N_BOOT} | seed: {SEED}')

## 2 · From event slugs to games

The rules below recognise explicit game-name prefixes or clear game names in event titles. Familiar league prefixes such as `lec` are mapped to League of Legends. Ambiguous tournament brands and unrelated titles are not guessed. If explicit prefix and title evidence disagree, the event is marked ambiguous.

Counter-Strike 2 and CS:GO are kept separate. Call of Duty and Overwatch are labelled as **series**, because the saved text does not necessarily identify a particular release. Review the coverage and example unclassified events before interpreting the results.

In [ ]:
PREFIX_GAME = {
    'cs2': 'Counter-Strike 2', 'csgo': 'Counter-Strike: Global Offensive',
    'lol': 'League of Legends', 'lec': 'League of Legends', 'lck': 'League of Legends', 'lpl': 'League of Legends',
    'dota2': 'Dota 2', 'val': 'Valorant', 'valorant': 'Valorant', 'vct': 'Valorant',
    'hok': 'Honor of Kings', 'mlbb': 'Mobile Legends: Bang Bang',
    'r6siege': 'Rainbow Six Siege', 'codmw': 'Call of Duty (series)',
    'sc2': 'StarCraft II', 'rl': 'Rocket League', 'ow': 'Overwatch (series)',
    'geoguessr': 'GeoGuessr',
}
TITLE_RULES = {
    'Counter-Strike 2': r'\b(?:cs2|counter[\s-]*strike\s*2)\b',
    'Counter-Strike: Global Offensive': r'\b(?:cs\s*:\s*go|csgo|counter[\s-]*strike\s*:\s*global offensive)\b',
    'League of Legends': r'\b(?:league of legends|lol)\b',
    'Dota 2': r'\bdota[\s-]*2\b',
    'Valorant': r'\bvalorant\b',
    'Honor of Kings': r'\bhonor of kings\b',
    'Mobile Legends: Bang Bang': r'\b(?:mobile legends|mlbb)\b',
    'Rainbow Six Siege': r'\brainbow six siege\b',
    'Call of Duty (series)': r'\bcall of duty\b',
    'StarCraft II': r'\bstarcraft\s*(?:2|ii)\b',
    'Rocket League': r'\brocket league\b',
    'Overwatch (series)': r'\boverwatch\b',
    'GeoGuessr': r'\bgeoguessr\b',
}
compiled_title_rules = {game: re.compile(pattern, re.IGNORECASE) for game, pattern in TITLE_RULES.items()}
UNCLASSIFIED = 'Unclassified / ambiguous'

def classify_game(slug, title):
    slug, title = str(slug or '').lower(), str(title or '')
    prefix = slug.split('-', 1)[0]
    candidates = set()
    if prefix in PREFIX_GAME:
        candidates.add(PREFIX_GAME[prefix])
    if slug.startswith('dota-2-'):
        candidates.add('Dota 2')
    candidates.update(game for game, regex in compiled_title_rules.items() if regex.search(title))
    if len(candidates) == 1:
        return next(iter(candidates)), 'explicit prefix' if prefix in PREFIX_GAME or slug.startswith('dota-2-') else 'explicit title'
    return UNCLASSIFIED, 'conflicting evidence' if len(candidates) > 1 else 'no explicit game'

display(pd.DataFrame([{'slug prefix': k, 'game': v} for k, v in PREFIX_GAME.items()]))

In [ ]:
full = pd.read_pickle(CACHE)
esports = full.loc[full['sport'].eq('Esports')].copy()
if esports.duplicated(['market_id', 'probability_bin']).any():
    raise ValueError('A market appears twice in a probability bin; inspect the cache.')
target_events = set(esports['event_id'].dropna().astype(str))
event_records = {}
with EVENTS.open(encoding='utf-8') as source:
    for line in source:
        if not line.strip():
            continue
        event = json.loads(line)
        event_id = str(event.get('id', ''))
        if event_id in target_events and event_id not in event_records:
            game, rule_source = classify_game(event.get('slug'), event.get('title'))
            event_records[event_id] = {'event_id': event_id, 'event_slug': event.get('slug', ''),
                                       'event_title': event.get('title', ''), 'game': game, 'game_rule': rule_source}
if source_fingerprint(source_files) != fingerprint:
    raise RuntimeError('A source file changed during loading; rerun with a consistent snapshot.')
event_metadata = pd.DataFrame(event_records.values())
if event_metadata.empty:
    raise ValueError('No saved event metadata matched the esports subset.')
esports = esports.merge(event_metadata, on='event_id', how='left', validate='many_to_one')
esports['game'] = esports['game'].fillna(UNCLASSIFIED)
esports['game_rule'] = esports['game_rule'].fillna('event metadata missing')
esports['residual'] = esports['actual'] - esports['probability']
esports['brier'] = (esports['probability'] - esports['actual']) ** 2

coverage = (esports.groupby(['game', 'game_rule'], dropna=False)
            .agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'),
                 market_bin_rows=('market_id', 'size')).reset_index()
            .sort_values('markets', ascending=False))
print(f'Esports subset: {esports.market_id.nunique():,} markets; {esports.event_id.nunique():,} linked events; {len(esports):,} market–bin rows.')
display(coverage.reset_index(drop=True))
print('A reproducible audit sample of unclassified events (not a sample limit on analysis):')
unclassified_events = event_metadata.loc[event_metadata['game'].eq(UNCLASSIFIED), ['event_slug', 'event_title', 'game_rule']]
display(unclassified_events.sample(n=min(20, len(unclassified_events)), random_state=SEED).reset_index(drop=True))

## 3 · Descriptive game comparison

For each game and probability bin, compare average probability $\bar p_b$ to the observed event frequency $\bar y_b$. Each market is counted once within a bin. Counts summed across bins can exceed the number of unique markets.

The Brier score is $\frac{1}{n}\sum_i(p_i-y_i)^2$. It reflects more than calibration: games with different probability distributions can have different Brier scores even when both are well calibrated. The bin-weighted absolute calibration gap below is descriptive, depends on the chosen bins, and is estimated on the same data.

Games enter the comparative analysis only if they have at least **30 distinct event IDs and 100 markets**. All identified games and exclusions remain visible in the coverage table above.

In [ ]:
game_sizes = (esports.loc[esports['game'].ne(UNCLASSIFIED) & esports['event_id'].notna()]
              .groupby('game').agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'))
              .sort_values('markets', ascending=False))
GAMES = game_sizes.loc[(game_sizes['events'] >= MIN_EVENTS) & (game_sizes['markets'] >= MIN_MARKETS)].index.tolist()
if len(GAMES) < 2:
    raise ValueError('Fewer than two games meet the market/event thresholds.')
analysis = esports.loc[esports['game'].isin(GAMES) & esports['event_id'].notna()].copy()
game_summary = (analysis.groupby('game').agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'),
                market_bin_rows=('market_id', 'size'), mean_probability=('probability', 'mean'),
                observed_rate=('actual', 'mean'), mean_y_minus_p=('residual', 'mean'), brier=('brier', 'mean')))
group_bins = (analysis.groupby(['game', 'probability_bin'])
              .agg(n=('market_id', 'size'), mean_p=('probability', 'mean'), observed=('actual', 'mean')))
group_bins['weighted_abs_gap'] = group_bins['n'] * (group_bins['observed'] - group_bins['mean_p']).abs()
ece = (group_bins.groupby('game')['weighted_abs_gap'].sum() / group_bins.groupby('game')['n'].sum()).rename('descriptive_binned_abs_gap')
game_summary = game_summary.join(ece).loc[GAMES]
display(game_summary)

## 4 · Calibration curves and event-cluster bootstrap

We resample saved **event IDs**, carrying all markets and probability-bin rows for each sampled event. For every resample, we recompute each game's bin event rate. The 2.5th and 97.5th percentiles form approximate 95% intervals.

The orange line is a descriptive weighted linear regression through the bin means, with weights proportional to bin counts. It is distinct from the individual-row logistic calibration model below. Treat the intervals cautiously if related games or markets have different event IDs.

In [ ]:
clusters, event_ids = pd.factorize(analysis['event_id'], sort=True)
game_codes = pd.Categorical(analysis['game'], categories=GAMES).codes
bin_codes = analysis['probability_bin'].to_numpy(dtype=int)
y = analysis['actual'].to_numpy(dtype=float)
p = analysis['probability'].to_numpy(dtype=float)
n_events, n_games = len(event_ids), len(GAMES)
cell_codes = game_codes * N_BINS + bin_codes

def event_matrix(values, cols):
    return sparse.coo_matrix((values, (clusters, cols)), shape=(n_events, n_games * N_BINS)).tocsr()

counts = event_matrix(np.ones(len(analysis)), cell_codes)
successes = event_matrix(y, cell_codes)
prob_sums = event_matrix(p, cell_codes)
obs_n = np.asarray(counts.sum(axis=0)).ravel()
obs_y = np.asarray(successes.sum(axis=0)).ravel()
obs_p = np.asarray(prob_sums.sum(axis=0)).ravel()
boot_rates = np.full((N_BOOT, n_games * N_BINS), np.nan)
boot_gaps = np.full((N_BOOT, n_games), np.nan)
rng = np.random.default_rng(SEED)
for b in range(N_BOOT):
    weights = np.bincount(rng.integers(0, n_events, size=n_events), minlength=n_events)
    n_star = np.asarray(counts.T @ weights).ravel()
    y_star = np.asarray(successes.T @ weights).ravel()
    p_star = np.asarray(prob_sums.T @ weights).ravel()
    boot_rates[b] = np.divide(y_star, n_star, out=np.full_like(y_star, np.nan), where=n_star > 0)
    game_n = n_star.reshape(n_games, N_BINS).sum(axis=1)
    game_resid = (y_star - p_star).reshape(n_games, N_BINS).sum(axis=1)
    boot_gaps[b] = np.divide(game_resid, game_n, out=np.full_like(game_resid, np.nan), where=game_n > 0)
    if (b + 1) % 100 == 0:
        print(f'Bootstrap {b + 1:,}/{N_BOOT:,}', flush=True)
print(f'Resampled {n_events:,} distinct saved event IDs per replicate.')

In [ ]:
rows = []
for j, game in enumerate(GAMES):
    for b in range(N_BINS):
        k = j * N_BINS + b
        if obs_n[k] == 0:
            continue
        draws = boot_rates[:, k]
        draws = draws[np.isfinite(draws)]
        lo, hi = np.quantile(draws, [.025, .975]) if len(draws) else (np.nan, np.nan)
        rows.append({'game': game, 'bin': b, 'n': int(obs_n[k]), 'mean_p': obs_p[k] / obs_n[k],
                     'observed': obs_y[k] / obs_n[k], 'ci_low': lo, 'ci_high': hi})
calibration_table = pd.DataFrame(rows)

ncols = 3
nrows = int(np.ceil(n_games / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 3.8 * nrows), sharex=True, sharey=True, constrained_layout=True, squeeze=False)
grid = np.linspace(0, 1, 101)
for ax, game in zip(axes.flat, GAMES):
    group = calibration_table.loc[calibration_table['game'].eq(game)]
    ax.plot([0, 1], [0, 1], '--', color='0.5', lw=1)
    if len(group) >= 2:
        x = group['mean_p'].to_numpy()
        weights = np.sqrt(group['n'].to_numpy())
        X = np.column_stack([np.ones(len(x)), x])
        intercept, slope = np.linalg.lstsq(X * weights[:, None], group['observed'].to_numpy() * weights, rcond=None)[0]
        ax.plot(grid, intercept + slope * grid, color='#b8642c', lw=1.2)
    ax.vlines(group['mean_p'], group['ci_low'], group['ci_high'], color='#386a9e', lw=1.2)
    ax.scatter(group['mean_p'], group['observed'], color='#386a9e', s=18, zorder=3)
    ax.set(title=f"{game}\nmarkets = {game_summary.loc[game, 'markets']:,}; events = {game_summary.loc[game, 'events']:,}",
           xlim=(0, 1), ylim=(0, 1), aspect='equal')
for ax in axes.flat[n_games:]:
    ax.axis('off')
for ax in axes[:, 0]:
    ax.set_ylabel('Observed event frequency')
for ax in axes[-1, :]:
    if ax.axison:
        ax.set_xlabel('Predicted probability')
handles = [Line2D([0], [0], color='0.5', ls='--', label='Perfect calibration'),
           Line2D([0], [0], color='#b8642c', label='Weighted bin regression'),
           Line2D([0], [0], color='#386a9e', marker='o', ls='', label='Bin mean; event-bootstrap 95% interval')]
fig.legend(handles=handles, loc='outside lower center', ncols=3)
fig.suptitle('Esports calibration by game · hours 2–24', fontsize=14)
plt.show()
display(calibration_table)

## 5 · Do games differ in average calibration gap?

For each game, define $\Delta_g=\operatorname{mean}(Y-p)$. We compare each eligible game with the largest game by market count, using event-cluster bootstrap intervals for $\Delta_g-\Delta_{\mathrm{ref}}$. Centred bootstrap p-values are approximate and exploratory. Benjamini–Hochberg (BH) and the more conservative Benjamini–Yekutieli (BY) adjustments account for the family of game comparisons.

These tests concern the **average** gap and can miss opposite errors that cancel across probability bins. They are associations, not evidence that the game itself causes a difference: market type, probability distribution, date, volume and data coverage may differ by game. With 500 replicates the minimum p-value is $1/501$; increase the count for final reporting.

In [ ]:
total_n = obs_n.reshape(n_games, N_BINS).sum(axis=1)
total_residual = (obs_y - obs_p).reshape(n_games, N_BINS).sum(axis=1)
gaps = total_residual / total_n
ref_idx = 0
REFERENCE = GAMES[ref_idx]
gap_rows, contrast_rows = [], []
for j, game in enumerate(GAMES):
    draws = boot_gaps[:, j]
    draws = draws[np.isfinite(draws)]
    lo, hi = np.quantile(draws, [.025, .975])
    gap_rows.append({'game': game, 'mean_y_minus_p': gaps[j], 'ci_low': lo, 'ci_high': hi})
    if j != ref_idx:
        paired = boot_gaps[:, j] - boot_gaps[:, ref_idx]
        paired = paired[np.isfinite(paired)]
        if len(paired) >= N_BOOT * .9:
            diff = gaps[j] - gaps[ref_idx]
            lo_diff, hi_diff = np.quantile(paired, [.025, .975])
            p_boot = (1 + np.count_nonzero(np.abs(paired - diff) >= abs(diff))) / (len(paired) + 1)
            contrast_rows.append({'game': game, 'reference': REFERENCE, 'gap_difference': diff,
                                  'ci_low': lo_diff, 'ci_high': hi_diff, 'p_boot': p_boot})
gap_table = pd.DataFrame(gap_rows).set_index('game').loc[GAMES]
contrasts = pd.DataFrame(contrast_rows)
if not contrasts.empty:
    contrasts['p_BH'] = false_discovery_control(contrasts['p_boot'].to_numpy(), method='bh')
    contrasts['p_BY'] = false_discovery_control(contrasts['p_boot'].to_numpy(), method='by')

fig, ax = plt.subplots(figsize=(8.5, max(3.5, n_games * .42)), constrained_layout=True)
yp = np.arange(n_games)
ax.axvline(0, color='0.5', ls='--', lw=1)
ax.hlines(yp, gap_table['ci_low'], gap_table['ci_high'], color='#386a9e', lw=1.5)
ax.scatter(gap_table['mean_y_minus_p'], yp, color='#386a9e', s=25, zorder=3)
ax.set_yticks(yp, GAMES)
ax.invert_yaxis()
ax.set(xlabel='Mean observed outcome minus predicted probability', title='Average calibration gap by game · event-bootstrap 95% intervals')
plt.show()
print('Reference game:', REFERENCE)
display(contrasts.sort_values('p_BY').reset_index(drop=True) if not contrasts.empty else contrasts)

## 6 · Final-volume stratification

Within each game, split unique markets with usable final volume into ranked low, middle and high tertiles. A market keeps one tier even if it appears in several probability bins. These are **descriptive associations**: total final volume is not guaranteed to be known at the time of a price observation, and other game characteristics may differ.

In [ ]:
market_volume = analysis.drop_duplicates('market_id')[['market_id', 'game', 'volumeNum']].copy()
market_volume['volumeNum'] = pd.to_numeric(market_volume['volumeNum'], errors='coerce')
market_volume['volume_tier'] = 'Missing'
for game in GAMES:
    mask = market_volume['game'].eq(game) & market_volume['volumeNum'].ge(0)
    if mask.sum() >= 3:
        market_volume.loc[mask, 'volume_tier'] = pd.qcut(
            market_volume.loc[mask, 'volumeNum'].rank(method='first'), 3,
            labels=['Low', 'Middle', 'High']
        ).astype(str).to_numpy()
analysis['volume_tier'] = analysis['market_id'].map(market_volume.set_index('market_id')['volume_tier'])
volume_summary = (analysis.loc[analysis['volume_tier'].ne('Missing')]
    .groupby(['game', 'volume_tier'], observed=True)
    .agg(markets=('market_id', 'nunique'), events=('event_id', 'nunique'), market_bin_rows=('market_id', 'size'),
         mean_probability=('probability', 'mean'), observed_rate=('actual', 'mean'),
         mean_y_minus_p=('residual', 'mean'), brier=('brier', 'mean')).reset_index())
display(volume_summary.sort_values(['game', 'volume_tier']).reset_index(drop=True))

## 7 · Game-specific logistic calibration

For each game we fit $\operatorname{logit}\Pr(Y=1)=\alpha_g+\beta_g\operatorname{logit}(p)$ to market–bin rows. The ideal intercept and slope are $0$ and $1$. Estimates are descriptive; they are not event-adjusted coefficient confidence intervals. Probabilities are clipped to $[10^{-6},1-10^{-6}]$ before taking logits, and each market can contribute to multiple bins.

In [ ]:
logistic_rows = []
for game in GAMES:
    group = analysis.loc[analysis['game'].eq(game)]
    prob = np.clip(group['probability'].to_numpy(dtype=float), EPS, 1 - EPS)
    target = group['actual'].to_numpy(dtype=int)
    if len(np.unique(target)) < 2:
        continue
    z = np.log(prob / (1 - prob)).reshape(-1, 1)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=500).fit(z, target)
    logistic_rows.append({'game': game, 'intercept': model.intercept_[0], 'slope': model.coef_[0, 0],
                          'market_bin_rows': len(group), 'markets': group['market_id'].nunique(),
                          'events': group['event_id'].nunique(), 'converged': int(model.n_iter_[0]) < 500})
logistic_summary = pd.DataFrame(logistic_rows).set_index('game').loc[GAMES]
display(logistic_summary)

### Visualising the logistic calibration models

For each game, the fitted curve transforms a market probability $p$ into the fitted event frequency

$$
\widehat{P}(Y=1\mid p)=\frac{1}{1+\exp[-\{\hat\alpha_g+\hat\beta_g\operatorname{logit}(p)\}]}.
$$

The blue points and whiskers show observed probability-bin rates and their event-bootstrap 95% intervals. The orange line is the **individual-row logistic regression** from Section 7, not the weighted regression through bin means shown earlier. The whiskers describe the observed bin rates, not uncertainty in the fitted logistic line. All these game-specific curves are fitted and shown on the same data, so they are descriptive, not out-of-sample validation.

In [ ]:
from scipy.special import expit, logit

available_games = [game for game in GAMES if game in logistic_summary.index]
ncols = 3
nrows = int(np.ceil(len(available_games) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 3.8 * nrows),
                         sharex=True, sharey=True, constrained_layout=True, squeeze=False)
probability_grid = np.linspace(EPS, 1 - EPS, 499)
logit_grid = logit(probability_grid)
for ax, game in zip(axes.flat, available_games):
    row = logistic_summary.loc[game]
    bins = calibration_table.loc[calibration_table['game'].eq(game)]
    fitted = expit(float(row['intercept']) + float(row['slope']) * logit_grid)
    ax.plot([0, 1], [0, 1], '--', color='0.5', lw=1)
    ax.plot(probability_grid, fitted, color='#b8642c', lw=1.5)
    ax.vlines(bins['mean_p'], bins['ci_low'], bins['ci_high'], color='#386a9e', lw=1.2)
    ax.scatter(bins['mean_p'], bins['observed'], color='#386a9e', s=18, zorder=3)
    ax.set(title=f"{game}\nα = {row['intercept']:.2f}; β = {row['slope']:.2f}; markets = {int(row['markets']):,}",
           xlim=(0, 1), ylim=(0, 1), aspect='equal')
for ax in axes.flat[len(available_games):]:
    ax.axis('off')
for ax in axes[:, 0]:
    ax.set_ylabel('Observed or fitted event frequency')
for ax in axes[-1, :]:
    if ax.axison:
        ax.set_xlabel('Market probability')
handles = [
    Line2D([0], [0], color='0.5', ls='--', label='Perfect calibration'),
    Line2D([0], [0], color='#b8642c', label='Fitted logistic curve'),
    Line2D([0], [0], color='#386a9e', marker='o', ls='', label='Observed bin rate; event-bootstrap 95% interval'),
]
fig.legend(handles=handles, loc='outside lower center', ncols=3)
fig.suptitle('Game-specific logistic calibration · hours 2–24', fontsize=14)
plt.show()

The next plot makes the departures from the diagonal easier to compare for the four games with the most markets. A value of **+5 percentage points** means the fitted event frequency is five percentage points higher than the market probability. These differences are fitted on the same observations and do not have confidence intervals here.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.7), constrained_layout=True)
comparison_grid = np.linspace(.01, .99, 199)
styles = ['-', '--', '-.', ':']
for game, style in zip(available_games[:4], styles):
    row = logistic_summary.loc[game]
    fitted = expit(float(row['intercept']) + float(row['slope']) * logit(comparison_grid))
    ax.plot(comparison_grid, 100 * (fitted - comparison_grid), lw=1.7, ls=style, label=game)
ax.axhline(0, color='0.45', lw=1)
ax.set(xlim=(0, 1), xlabel='Market probability',
       ylabel='Fitted frequency − market probability\n(percentage points)',
       title='Logistic calibration differences · four largest games')
ax.legend(loc='best')
plt.show()

## 8 · Chronological game-aware validation and ridge shrinkage

We compare the original price with a global logistic calibration and a game-interaction logistic model with ridge shrinkage. The latter uses game indicators and interactions with log-odds. Two expanding, event-grouped chronological training folds select `C`; the final event groups are held out.

The split uses market `endDate`, excludes events spanning the cutoff, and adds a 24-hour embargo before test observations. Final resolutions may have occurred after recorded `endDate`, and prices aggregate a trajectory rather than a fixed pre-game instant. Therefore, the comparison is a **retrospective sensitivity analysis**, not a leak-free live forecasting or betting backtest.

In [ ]:
dated = analysis.loc[analysis['end_date'].notna()].copy()
spans = dated.groupby('event_id')['end_date'].agg(event_min='min', event_max='max').dropna()
cutoff = spans['event_max'].quantile(.80)
train_events = spans.index[spans['event_max'] < cutoff]
test_events = spans.index[spans['event_min'] >= cutoff + pd.Timedelta(hours=24)]
excluded_events = spans.index.difference(train_events.union(test_events))
training_candidates = dated.loc[dated['event_id'].isin(train_events)]
model_games = training_candidates.groupby('game')['market_id'].nunique().nlargest(7).index.tolist()
model_data = dated.loc[dated['game'].isin(model_games) & dated['event_id'].isin(train_events.union(test_events))].copy()
train_mask = model_data['event_id'].isin(train_events).to_numpy()
test_mask = model_data['event_id'].isin(test_events).to_numpy()
assert set(model_data.loc[train_mask, 'event_id']).isdisjoint(model_data.loc[test_mask, 'event_id'])
assert model_data.loc[train_mask, 'end_date'].max() < model_data.loc[test_mask, 'end_date'].min() - pd.Timedelta(hours=24)
print(f'Cutoff: {cutoff}; boundary/embargo events excluded: {len(excluded_events):,}')
print(f'Training rows: {train_mask.sum():,}; test rows: {test_mask.sum():,}; model games: {model_games}')

model_p = np.clip(model_data['probability'].to_numpy(dtype=float), EPS, 1 - EPS)
model_y = model_data['actual'].to_numpy(dtype=int)
z = np.log(model_p / (1 - model_p))
X_base = z.reshape(-1, 1)
categories = pd.Categorical(model_data['game'], categories=model_games)
dummies = pd.get_dummies(categories, drop_first=True, dtype=float).to_numpy()
X_game = np.column_stack([z, dummies, z[:, None] * dummies])
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    mle = LogisticRegression(penalty=None, solver='lbfgs', max_iter=500).fit(X_base[train_mask], model_y[train_mask])

train_spans = spans.loc[train_events]
fold_cuts = [train_spans['event_max'].quantile(.50), train_spans['event_max'].quantile(.75), cutoff]
folds = []
for lo, hi in zip(fold_cuts[:-1], fold_cuts[1:]):
    fit_events = train_spans.index[train_spans['event_max'] < lo]
    val_events = train_spans.index[(train_spans['event_min'] >= lo + pd.Timedelta(hours=24)) & (train_spans['event_max'] < hi)]
    fit_mask = model_data['event_id'].isin(fit_events).to_numpy()
    val_mask = model_data['event_id'].isin(val_events).to_numpy()
    if not fit_mask.any() or not val_mask.any() or len(np.unique(model_y[fit_mask])) < 2:
        raise ValueError('A chronological fold has insufficient events or outcomes; adjust cutoffs.')
    assert model_data.loc[fit_mask, 'end_date'].max() < model_data.loc[val_mask, 'end_date'].min() - pd.Timedelta(hours=24)
    folds.append((fit_mask, val_mask))
    print(f'Fold: training rows {fit_mask.sum():,}; validation rows {val_mask.sum():,}')

cv_rows = []
for C in [0.01, 0.1, 1.0]:
    for fold, (fit_mask, val_mask) in enumerate(folds, 1):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, solver='lbfgs', max_iter=500))
        model.fit(X_game[fit_mask], model_y[fit_mask])
        pred = model.predict_proba(X_game[val_mask])[:, 1]
        cv_rows.append({'C': C, 'fold': fold, 'validation_rows': int(val_mask.sum()),
                        'log_loss': log_loss(model_y[val_mask], pred, labels=[0, 1])})
cv = pd.DataFrame(cv_rows)
scores = (cv.assign(weighted_loss=cv['log_loss'] * cv['validation_rows'])
          .groupby('C')[['weighted_loss', 'validation_rows']].sum())
scores['mean_log_loss'] = scores['weighted_loss'] / scores['validation_rows']
best_C = float(scores['mean_log_loss'].idxmin())
display(cv)
print(f'Selected C: {best_C}')

ridge = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, solver='lbfgs', max_iter=500))
ridge.fit(X_game[train_mask], model_y[train_mask])
predictions = {'Original market probability': model_p[test_mask],
               'Global logistic calibration': mle.predict_proba(X_base[test_mask])[:, 1],
               'Game-interaction ridge': ridge.predict_proba(X_game[test_mask])[:, 1]}
evaluation = pd.DataFrame([
    {'method': name, 'test_market_bin_rows': int(test_mask.sum()),
     'test_markets': model_data.loc[test_mask, 'market_id'].nunique(),
     'test_events': model_data.loc[test_mask, 'event_id'].nunique(),
     'brier': brier_score_loss(model_y[test_mask], pred),
     'log_loss': log_loss(model_y[test_mask], pred, labels=[0, 1])}
    for name, pred in predictions.items()
]).sort_values('log_loss')
display(evaluation.reset_index(drop=True))

## 9 · Interpretation and limitations

- The unit is a market–probability-bin pair, not an independent trade. Different bins can contain the same market; event resampling groups related markets only to the extent that the event IDs are correct.
- Game labels come from explicit slug/title rules applied **after** the event-sport classifier. Inspect the unclassified sample and audit a random sample of assigned labels; neither classifier is verified ground truth.
- Comparisons are descriptive. A different game can have different market types, price distributions, seasons and trading activity. Mean Brier score and the mean calibration gap should not be read as a causal ranking of games.
- Hourly VWAPs are relative to `endDate`, which may not be the event start; some outcomes may be known before that date. The current cache does not establish fixed pre-game forecasts or resolution-time availability.
- Final volume is a retrospective snapshot; historical hourly liquidity and spread cannot be recovered from the final metadata. Some trade histories may be truncated at the API limit.
- Sensitivity checks: audit event/game labels; compare the 24-hour version; exclude potentially truncated histories if flags can be linked; use a fixed pre-event time where reliable event start times exist; and repeat with different bin widths and more bootstrap replicates.

### References

- Efron, B. and Hastie, T. *Computer Age Statistical Inference*. Chapters 4, 8, 10–12 and 15. https://hastie.su.domains/CASI/
- Murphy, A. H. (1973). “A New Vector Partition of the Probability Score.” *Journal of Applied Meteorology*. https://doi.org/10.1175/1520-0450(1973)012<0595:ANVPOT>2.0.CO;2